# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 10 — Scaffold-disjoint cross-validation

The comparison between models has been carried out over fifteen scaffold partitions. Those
partitions turned out not to be independent: their test sets overlap at a Jaccard index of
0.63 against the 0.05 expected of independent draws, the fifteen of them together cover only
937 of the 7,255 compounds, and the design carries the weight of roughly one and a quarter
independent observations rather than fifteen. The cause is the splitter, which orders
scaffold groups by descending size and so sends the same large groups to training every time.

Cross-validation removes the problem rather than measuring it. Scaffolds are assigned to ten
folds, each fold serves once as the test set, and every compound receives exactly one
out-of-fold prediction. The result is a single held-out prediction for the whole dataset, on
which models can be compared by a paired bootstrap over compounds. No omnibus test across
partitions is needed, and the question of how many independent partitions there are does not
arise.

**Contents**

1. Environment and shared definitions
2. Data and folds
3. Fitting each model on each fold
4. Out-of-fold comparison
5. Representation, learner and family
6. Bond features
7. Comparison with the earlier design
8. Summary

*Approximate runtime: five to six hours on a T4. Each fold checkpoints.*

## 1. Environment and shared definitions

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 50.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/models"
CKPT = f"{WORK}/ckpt_part10"
for d in (R, CKPT): os.makedirs(d, exist_ok=True)
print("checkpoints:", CKPT)

Mounted at /content/drive
checkpoints: /content/drive/MyDrive/ames_pipeline/ckpt_part10


The definitions below are those used in Parts 2 to 9, reproduced here so that this notebook
runs on a fresh session. The training routine is unchanged, so a difference in results can
only come from the partitioning.

In [3]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


## 2. Data and folds

Scaffolds are shuffled and dealt into ten folds of roughly equal size. Because assignment is
by scaffold, no scaffold appears in both the training and the test part of any fold, and
because each fold is held out exactly once, every compound is predicted exactly once from a
model that never saw its scaffold.

Within each fold's training portion, a tenth is set aside for validation, which the graph
models use for early stopping and the gradient-boosted models for their stopping rounds. That
validation set is also scaffold-disjoint from the fold's test set.

In [4]:
import sys, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from collections import defaultdict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef, recall_score)

SEED, N_FOLDS, SEEDS = 42, 10, [42, 43, 44]
HP_GNN     = json.load(open(f"{R}/hpo_gnn.json"))
HP_CLASSIC = json.load(open(f"{R}/hpo_classic.json"))
HP_DESC    = json.load(open(f"{R}/hpo_descriptor.json"))
HP_NOEDGE  = json.load(open(f"{R}/hpo_gnn_noedge.json"))

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
t0 = time.time()
df["scaffold"] = df["canonical"].map(safe_scaffold)
print(f"{len(df)} compounds, {df.scaffold.nunique()} scaffolds ({time.time()-t0:.0f} s)")

def scaffold_folds(frame, n_folds=N_FOLDS, seed=SEED):
    """Deal scaffold groups into folds, largest first into the smallest fold."""
    groups = defaultdict(list)
    for i, s in zip(frame.index, frame["scaffold"]):
        groups[s].append(i)
    gs = list(groups.values())
    rng = np.random.default_rng(seed)
    rng.shuffle(gs)
    gs.sort(key=len, reverse=True)
    folds = [[] for _ in range(n_folds)]
    for g in gs:
        folds[int(np.argmin([len(f) for f in folds]))] += g
    return folds

FOLDS = scaffold_folds(df)
sizes = [len(f) for f in FOLDS]
print(f"fold sizes {sizes}, total {sum(sizes)}")
assert sum(sizes) == len(df) and len(set(sum(FOLDS, []))) == len(df)
scafs = [set(df.loc[f, "scaffold"]) for f in FOLDS]
for a in range(N_FOLDS):
    for b in range(a+1, N_FOLDS):
        assert not (scafs[a] & scafs[b]), "folds share a scaffold"
print("folds are scaffold-disjoint; every compound appears exactly once")
pd.DataFrame({"compound": df["canonical"],
              "fold": [next(k for k, f in enumerate(FOLDS) if i in set(f))
                       for i in df.index]}).to_csv(f"{R}/fold_assignment.csv", index=False)

7255 compounds, 1577 scaffolds (2 s)
fold sizes [1260, 1135, 608, 608, 608, 608, 607, 607, 607, 607], total 7255
folds are scaffold-disjoint; every compound appears exactly once


## 3. Fitting each model on each fold

Six models, three seeds each, ten folds. Every configuration is the one selected by the
hyperparameter search reported earlier; nothing is re-tuned, because re-tuning inside the
cross-validation would confound the change of partitioning with a change of configuration.

In [5]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(names)
DP = f"{R}/rdkit2d_descriptors.npz"
if os.path.exists(DP):
    z = np.load(DP, allow_pickle=True); DESC, SM = z["desc"], list(z["smiles"])
else:
    DESC = np.nan_to_num(np.array([list(calc.CalcDescriptors(Chem.MolFromSmiles(s)))
                                   for s in df["canonical"]], dtype=float),
                         nan=0.0, posinf=0.0, neginf=0.0)
    SM = list(df["canonical"]); np.savez(DP, desc=DESC, smiles=np.array(SM, dtype=object))
DIDX = {s: i for i, s in enumerate(SM)}
print(f"descriptors {DESC.shape}")

def inner_split(train_idx, seed):
    """Hold out a scaffold-disjoint tenth of the fold's training portion."""
    sub = df.loc[train_idx]
    groups = defaultdict(list)
    for i, s in zip(sub.index, sub["scaffold"]):
        groups[s].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    rng.shuffle(gs)
    target = int(0.1 * len(sub)); va = []
    for g in gs:
        if len(va) + len(g) <= target: va += g
    va_set = set(va)
    return [i for i in train_idx if i not in va_set], va

MODELS_LIST = ["XGBoost", "RandomForest", "XGBoost_RDKit2D", "GCN", "GIN", "GAT",
               "GIN_noedge", "GAT_noedge"]

def fit_fold(k):
    te_idx = FOLDS[k]
    tr_all = [i for j, f in enumerate(FOLDS) if j != k for i in f]
    tr_idx, va_idx = inner_split(tr_all, SEED + k)
    d_tr, d_va, d_te = df.loc[tr_idx], df.loc[va_idx], df.loc[te_idx]
    y_tr, y_va, y_te = (d_tr.label.values, d_va.label.values, d_te.label.values)
    Xtr, _ = build_X(d_tr); Xva, _ = build_X(d_va); Xte, _ = build_X(d_te)
    Dtr = DESC[[DIDX[s] for s in d_tr.canonical]]
    Dva = DESC[[DIDX[s] for s in d_va.canonical]]
    Dte = DESC[[DIDX[s] for s in d_te.canonical]]
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    out = {"fold": k, "test_index": list(te_idx), "y_true": y_te.tolist()}

    for name, Xa, Xb, Xc, cfg in [("XGBoost", Xtr, Xva, Xte, HP_CLASSIC["XGBoost"]),
                                  ("XGBoost_RDKit2D", Dtr, Dva, Dte, HP_DESC)]:
        ps = []
        for s in SEEDS:
            m = XGBClassifier(n_estimators=1500, **cfg, scale_pos_weight=spw,
                              eval_metric="auc", early_stopping_rounds=50,
                              tree_method="hist", random_state=s, n_jobs=-1)
            m.fit(Xa, y_tr, eval_set=[(Xb, y_va)], verbose=False)
            ps.append(m.predict_proba(Xc)[:, 1])
        out[name] = np.mean(ps, 0).tolist()
    ps = []
    for s in SEEDS:
        m = RandomForestClassifier(**HP_CLASSIC["RandomForest"], class_weight="balanced",
                                   random_state=s, n_jobs=-1).fit(Xtr, y_tr)
        ps.append(m.predict_proba(Xte)[:, 1])
    out["RandomForest"] = np.mean(ps, 0).tolist()

    tr_set, va_set, te_set = build_dataset(d_tr), build_dataset(d_va), build_dataset(d_te)
    te_loader = DataLoader(te_set, batch_size=256)
    pw = torch.tensor([spw], dtype=torch.float, device=DEVICE)
    for name in ["GCN", "GIN", "GAT", "GIN_noedge", "GAT_noedge"]:
        kind = name.split("_")[0].lower()
        edge = not name.endswith("noedge")
        cfg = HP_GNN[kind] if edge else HP_NOEDGE[kind]
        ps = []
        for s in SEEDS:
            m, _ = train_gnn(kind, cfg, s, tr_set, va_set, pw, use_edge_attr=edge)
            _, p = evaluate(m, te_loader); ps.append(p)
            del m
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        out[name] = np.mean(ps, 0).tolist()
    return out

for k in range(N_FOLDS):
    path = f"{CKPT}/fold{k}.json"
    if os.path.exists(path):
        print(f"  fold {k}: loaded"); continue
    t0 = time.time()
    res = fit_fold(k)
    json.dump(res, open(path, "w"))
    aucs = {m: roc_auc_score(res["y_true"], res[m]) for m in MODELS_LIST}
    print(f"  fold {k}: n={len(res['y_true'])}  " +
          "  ".join(f"{m[:7]} {a:.3f}" for m, a in list(aucs.items())[:4]) +
          f"  ({(time.time()-t0)/60:.1f} min)")

descriptors (7255, 217)
  fold 0: n=1260  XGBoost 0.796  RandomF 0.764  XGBoost 0.813  GCN 0.835  (34.8 min)
  fold 1: n=1135  XGBoost 0.816  RandomF 0.807  XGBoost 0.818  GCN 0.823  (29.7 min)
  fold 2: n=608  XGBoost 0.869  RandomF 0.862  XGBoost 0.882  GCN 0.869  (34.7 min)
  fold 3: n=608  XGBoost 0.839  RandomF 0.844  XGBoost 0.842  GCN 0.801  (27.5 min)
  fold 4: n=608  XGBoost 0.858  RandomF 0.855  XGBoost 0.861  GCN 0.866  (33.6 min)
  fold 5: n=608  XGBoost 0.859  RandomF 0.853  XGBoost 0.883  GCN 0.849  (39.9 min)
  fold 6: n=607  XGBoost 0.833  RandomF 0.847  XGBoost 0.839  GCN 0.845  (35.1 min)
  fold 7: n=607  XGBoost 0.852  RandomF 0.861  XGBoost 0.871  GCN 0.854  (36.5 min)
  fold 8: n=607  XGBoost 0.872  RandomF 0.879  XGBoost 0.884  GCN 0.854  (28.8 min)
  fold 9: n=607  XGBoost 0.805  RandomF 0.822  XGBoost 0.845  GCN 0.835  (38.4 min)


## 4. Out-of-fold comparison

Assembling the folds gives one prediction per compound for each model, over the whole
dataset. Models are compared on that single vector by a bootstrap over compounds, resampled
jointly so that every model sees the same replicate. This replaces the Friedman and Nemenyi
tests, which assumed independent partitions.

In [6]:
oof = {m: np.full(len(df), np.nan) for m in MODELS_LIST}
y_all = np.full(len(df), np.nan)
pos = {i: p for p, i in enumerate(df.index)}
for k in range(N_FOLDS):
    r = json.load(open(f"{CKPT}/fold{k}.json"))
    rows = [pos[i] for i in r["test_index"]]
    y_all[rows] = r["y_true"]
    for m in MODELS_LIST:
        oof[m][rows] = r[m]
assert not np.isnan(y_all).any()
for m in MODELS_LIST:
    assert not np.isnan(oof[m]).any(), m
np.savez_compressed(f"{R}/oof_predictions.npz", y=y_all, **oof)
print(f"out-of-fold predictions for {len(y_all)} compounds, "
      f"{100*y_all.mean():.1f}% mutagenic")

def thr_on_folds(m):
    """Threshold chosen fold-wise on the training portion, applied out of fold."""
    grid = np.linspace(0.05, 0.95, 91)
    return grid[int(np.argmax([balanced_accuracy_score(y_all, (oof[m] >= t).astype(int))
                               for t in grid]))]

rows = []
for m in MODELS_LIST:
    p = oof[m]; t = thr_on_folds(m); yhat = (p >= t).astype(int)
    rows.append({"model": m, "ROC_AUC": roc_auc_score(y_all, p),
                 "PR_AUC": average_precision_score(y_all, p),
                 "balanced_acc": balanced_accuracy_score(y_all, yhat),
                 "sensitivity": recall_score(y_all, yhat, pos_label=1),
                 "specificity": recall_score(y_all, yhat, pos_label=0),
                 "MCC": matthews_corrcoef(y_all, yhat), "threshold": t})
cv = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
cv.to_csv(f"{R}/table53_crossval_accuracy.csv", index=False)
print()
print(cv.round(4).to_string(index=False))

out-of-fold predictions for 7255 compounds, 54.5% mutagenic

          model  ROC_AUC  PR_AUC  balanced_acc  sensitivity  specificity    MCC  threshold
     GAT_noedge   0.8543  0.8634        0.7775       0.7998       0.7551 0.5552       0.46
            GAT   0.8541  0.8655        0.7784       0.8332       0.7237 0.5618       0.44
            GIN   0.8454  0.8527        0.7827       0.7790       0.7863 0.5636       0.54
XGBoost_RDKit2D   0.8453  0.8637        0.7715       0.7221       0.8208 0.5414       0.53
     GIN_noedge   0.8437  0.8426        0.7742       0.8190       0.7294 0.5516       0.47
            GCN   0.8430  0.8421        0.7721       0.7793       0.7648 0.5430       0.45
        XGBoost   0.8324  0.8572        0.7552       0.7226       0.7878 0.5085       0.48
   RandomForest   0.8209  0.8474        0.7608       0.7170       0.8045 0.5199       0.49


In [7]:
N_BOOT = 10000
rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(y_all), (N_BOOT, len(y_all)))
boot = {m: np.empty(N_BOOT) for m in MODELS_LIST}
for b in range(N_BOOT):
    i = idx[b]; yb = y_all[i]
    if len(np.unique(yb)) < 2:
        for m in MODELS_LIST: boot[m][b] = np.nan
        continue
    for m in MODELS_LIST:
        boot[m][b] = roc_auc_score(yb, oof[m][i])

rows = []
for m in MODELS_LIST:
    v = boot[m][~np.isnan(boot[m])]
    rows.append({"model": m, "ROC_AUC": roc_auc_score(y_all, oof[m]),
                 "ci_low": np.percentile(v, 2.5), "ci_high": np.percentile(v, 97.5)})
ci = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
ci.to_csv(f"{R}/table54_crossval_ci.csv", index=False)
print(ci.round(4).to_string(index=False))

MAIN = ["XGBoost", "RandomForest", "XGBoost_RDKit2D", "GCN", "GIN", "GAT"]
rows = []
for a in range(len(MAIN)):
    for b_ in range(a+1, len(MAIN)):
        d = boot[MAIN[a]] - boot[MAIN[b_]]; d = d[~np.isnan(d)]
        rows.append({"model_a": MAIN[a], "model_b": MAIN[b_],
                     "difference": roc_auc_score(y_all, oof[MAIN[a]]) -
                                   roc_auc_score(y_all, oof[MAIN[b_]]),
                     "ci_low": np.percentile(d, 2.5), "ci_high": np.percentile(d, 97.5),
                     "p_two_sided": 2*min((d > 0).mean(), (d < 0).mean()),
                     "prob_a_better": float((d > 0).mean())})
pair = pd.DataFrame(rows)
p = pair.p_two_sided.values
order = np.argsort(p); kk = len(p); adj = np.empty(kk); run = 0.0
for r_, i_ in enumerate(order):
    run = max(run, (kk - r_) * p[i_]); adj[i_] = min(run, 1.0)
pair["p_holm"] = adj
pair["resolved"] = pair.p_holm < 0.05
pair.to_csv(f"{R}/table55_crossval_pairwise.csv", index=False)
print()
print(pair.round(4).to_string(index=False))

          model  ROC_AUC  ci_low  ci_high
     GAT_noedge   0.8543  0.8455   0.8631
            GAT   0.8541  0.8455   0.8627
            GIN   0.8454  0.8363   0.8545
XGBoost_RDKit2D   0.8453  0.8366   0.8541
     GIN_noedge   0.8437  0.8346   0.8530
            GCN   0.8430  0.8339   0.8518
        XGBoost   0.8324  0.8233   0.8414
   RandomForest   0.8209  0.8113   0.8303

        model_a         model_b  difference  ci_low  ci_high  p_two_sided  prob_a_better  p_holm  resolved
        XGBoost    RandomForest      0.0116  0.0067   0.0164       0.0000         1.0000  0.0000      True
        XGBoost XGBoost_RDKit2D     -0.0129 -0.0195  -0.0064       0.0002         0.0001  0.0014      True
        XGBoost             GCN     -0.0105 -0.0179  -0.0031       0.0042         0.0021  0.0210      True
        XGBoost             GIN     -0.0130 -0.0198  -0.0062       0.0000         0.0000  0.0000      True
        XGBoost             GAT     -0.0216 -0.0289  -0.0144       0.0000         0.00

## 5. Representation, learner and family

The decomposition reported earlier used paired differences across partitions. Here it uses
paired differences across compounds, on predictions that no model saw during fitting.

In [8]:
def contrast(a, b, label):
    da = boot[a] if isinstance(a, str) else np.nanmean([boot[x] for x in a], 0)
    db = boot[b] if isinstance(b, str) else np.nanmean([boot[x] for x in b], 0)
    d = da - db; d = d[~np.isnan(d)]
    pa = oof[a] if isinstance(a, str) else np.mean([oof[x] for x in a], 0)
    pb = oof[b] if isinstance(b, str) else np.mean([oof[x] for x in b], 0)
    return {"contrast": label,
            "difference": roc_auc_score(y_all, pa) - roc_auc_score(y_all, pb),
            "ci_low": np.percentile(d, 2.5), "ci_high": np.percentile(d, 97.5),
            "p_two_sided": 2*min((d > 0).mean(), (d < 0).mean())}

dec = pd.DataFrame([
    contrast("XGBoost_RDKit2D", "XGBoost", "Representation: ECFP4 to RDKit-2D"),
    contrast("RandomForest", "XGBoost", "Learner: XGBoost to random forest"),
    contrast(["GCN","GIN","GAT"], ["XGBoost","RandomForest","XGBoost_RDKit2D"],
             "Family: classical to graph"),
    contrast("GIN", "XGBoost_RDKit2D", "Best of each family"),
])
dec.to_csv(f"{R}/table56_crossval_decomposition.csv", index=False)
print(dec.round(4).to_string(index=False))

                         contrast  difference  ci_low  ci_high  p_two_sided
Representation: ECFP4 to RDKit-2D      0.0129  0.0064   0.0195       0.0002
Learner: XGBoost to random forest     -0.0116 -0.0164  -0.0067       0.0000
       Family: classical to graph      0.0105  0.0088   0.0204       0.0000
              Best of each family      0.0001 -0.0067   0.0068       0.9876


## 6. Bond features

Each ablated variant carries its own hyperparameter search, as before.

In [9]:
ab = pd.DataFrame([
    contrast("GIN", "GIN_noedge", "GIN with and without bond features"),
    contrast("GAT", "GAT_noedge", "GAT with and without bond features"),
])
ab.to_csv(f"{R}/table57_crossval_ablation.csv", index=False)
print(ab.round(4).to_string(index=False))

                          contrast  difference  ci_low  ci_high  p_two_sided
GIN with and without bond features      0.0017 -0.0029   0.0063       0.4712
GAT with and without bond features     -0.0002 -0.0043   0.0039       0.9256


## 7. Comparison with the earlier design

The same models, the same configurations and the same training routine, evaluated under two
partitioning schemes. Differences between the two columns are attributable to the design
rather than to the models.

In [10]:
old = pd.read_csv(f"{R}/benchmark_runs.csv")
old = old[(old.scheme == "scaffold") & (old.arm == "main")]
try:
    od = pd.read_csv(f"{R}/benchmark_runs_descriptor.csv")
except FileNotFoundError:
    od = None
rows = []
for m in MAIN:
    if m == "XGBoost_RDKit2D":
        prev = od.ROC_AUC.mean() if od is not None else np.nan
        prev_sd = od.ROC_AUC.std() if od is not None else np.nan
    else:
        s = old[old.model == m]
        prev, prev_sd = s.ROC_AUC.mean(), s.ROC_AUC.std()
    rows.append({"model": m, "fifteen_partitions": prev, "sd_across_partitions": prev_sd,
                 "cross_validation": roc_auc_score(y_all, oof[m]),
                 "difference": roc_auc_score(y_all, oof[m]) - prev})
cmpdf = pd.DataFrame(rows)
cmpdf.to_csv(f"{R}/table58_design_comparison.csv", index=False)
print(cmpdf.round(4).to_string(index=False))
print()
r_old = cmpdf.fifteen_partitions.rank(ascending=False)
r_new = cmpdf.cross_validation.rank(ascending=False)
from scipy.stats import spearmanr
print(f"rank correlation between the two designs: "
      f"{spearmanr(r_old, r_new).correlation:+.3f}")
print(f"spread of mean ROC-AUC: {cmpdf.fifteen_partitions.max()-cmpdf.fifteen_partitions.min():.4f} "
      f"under fifteen partitions, "
      f"{cmpdf.cross_validation.max()-cmpdf.cross_validation.min():.4f} under cross-validation")
try:
    pt = pd.read_csv(f"{R}/table41_partition_dependence.csv")
    e = pt[pt.scheme.str.contains("size-ordered")].iloc[0]
    print(f"\nthe earlier design carried {e.n_effective:.2f} effective independent "
          f"observations and covered {int(e.union_compounds)} compounds; cross-validation "
          f"covers {len(y_all)} with one held-out prediction each")
except FileNotFoundError:
    pass

          model  fifteen_partitions  sd_across_partitions  cross_validation  difference
        XGBoost              0.7801                0.0095            0.8324      0.0524
   RandomForest              0.7949                0.0078            0.8209      0.0259
XGBoost_RDKit2D              0.8021                0.0095            0.8453      0.0432
            GCN              0.7909                0.0082            0.8430      0.0520
            GIN              0.8024                0.0086            0.8454      0.0430
            GAT              0.7957                0.0140            0.8541      0.0584

rank correlation between the two designs: +0.657
spread of mean ROC-AUC: 0.0223 under fifteen partitions, 0.0332 under cross-validation

the earlier design carried 1.26 effective independent observations and covered 937 compounds; cross-validation covers 7255 with one held-out prediction each


## 8. Summary

In [11]:
print("out-of-fold ROC-AUC over", len(y_all), "compounds")
print(ci.round(4).to_string(index=False))
print("\nresolved pairwise differences after correction")
res = pair[pair.resolved]
print(res[["model_a","model_b","difference","p_holm"]].round(4).to_string(index=False)
      if len(res) else "  none")
print("\ncontrasts")
print(dec[["contrast","difference","ci_low","ci_high"]].round(4).to_string(index=False))
print("\nbond features")
print(ab[["contrast","difference","ci_low","ci_high"]].round(4).to_string(index=False))
print("\nWritten to", R)
for f in ["fold_assignment.csv","oof_predictions.npz","table53_crossval_accuracy.csv",
          "table54_crossval_ci.csv","table55_crossval_pairwise.csv",
          "table56_crossval_decomposition.csv","table57_crossval_ablation.csv",
          "table58_design_comparison.csv"]:
    print("  -", f)

import shutil
from google.colab import files
shutil.make_archive("/content/part10_results", "zip", R)
files.download("/content/part10_results.zip")

out-of-fold ROC-AUC over 7255 compounds
          model  ROC_AUC  ci_low  ci_high
     GAT_noedge   0.8543  0.8455   0.8631
            GAT   0.8541  0.8455   0.8627
            GIN   0.8454  0.8363   0.8545
XGBoost_RDKit2D   0.8453  0.8366   0.8541
     GIN_noedge   0.8437  0.8346   0.8530
            GCN   0.8430  0.8339   0.8518
        XGBoost   0.8324  0.8233   0.8414
   RandomForest   0.8209  0.8113   0.8303

resolved pairwise differences after correction
     model_a         model_b  difference  p_holm
     XGBoost    RandomForest      0.0116  0.0000
     XGBoost XGBoost_RDKit2D     -0.0129  0.0014
     XGBoost             GCN     -0.0105  0.0210
     XGBoost             GIN     -0.0130  0.0000
     XGBoost             GAT     -0.0216  0.0000
RandomForest XGBoost_RDKit2D     -0.0244  0.0000
RandomForest             GCN     -0.0221  0.0000
RandomForest             GIN     -0.0245  0.0000
RandomForest             GAT     -0.0332  0.0000
         GCN             GAT     -0.0111  0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>